# Bab 4: Regresi Linear dan Bayesian

Notebook pendamping buku *Pengenalan Pola dan Machine Learning: Pendekatan Probabilistik dengan Python*.

Jalankan sel dari atas ke bawah (**Runtime → Run all** di Colab). Sel pertama menyiapkan folder kode dan modul data sintetis; setiap sel berikutnya adalah satu listing dari buku, dengan nomor penanda `# (1)`, `# (2)`, … yang dijelaskan di bawah listing pada buku. Angka keluaran bisa sedikit berbeda dari buku bila versi pustakanya berbeda (lihat Lampiran A).

In [ ]:
# Menyiapkan folder kode/ dan modul pendukung bab ini
import os, pathlib
BERKAS = {
"bab04/basis_harga.py": "import numpy as np\nfrom data_harga import buat_data, fitur\n\nt, y = buat_data()\nlatih, uji = t < 156, t >= 156                     # 3 tahun latih, 2025 uji  #<1>\n\ndef rmse(Phi):\n    w, *_ = np.linalg.lstsq(Phi[latih], y[latih], rcond=None)  #<2>\n    galat = Phi @ w - y\n    return np.sqrt(np.mean(galat[latih]**2)), np.sqrt(np.mean(galat[uji]**2))\n\ns = (t - 104) / 104                                # skala [-1, 1] agar stabil\nmodel = {\n    \"polinom orde 9\":         np.vander(s, 10, increasing=True),   #<3>\n    \"Fourier saja\":           fitur(t, lebaran=False, tren=False),\n    \"Fourier + tren\":         fitur(t, lebaran=False),\n    \"Fourier + tren + Lebaran\": fitur(t),                          #<4>\n}\nprint(f\"{'model':26s} {'#param':>6s} {'RMSE latih':>11s} {'RMSE uji':>9s}\")\nfor nama, Phi in model.items():\n    r_l, r_u = rmse(Phi)\n    print(f\"{nama:26s} {Phi.shape[1]:6d} {r_l:11.2f} {r_u:9.2f}\")\n",
"bab04/bayes_harga.py": "import numpy as np\nfrom data_harga import buat_data, fitur, pola_harga\n\nt, y = buat_data()\nlatih = t < 156\nPhi, tl = fitur(t[latih]), y[latih]\nN, D = Phi.shape\n\nalpha, beta = 1.0, 1.0\nfor _ in range(100):                              # alpha, beta dari evidence  #<1>\n    S = np.linalg.inv(alpha * np.eye(D) + beta * Phi.T @ Phi)\n    m = beta * S @ Phi.T @ tl\n    gamma = D - alpha * np.trace(S)\n    alpha, beta = gamma / (m @ m), (N - gamma) / np.sum((tl - Phi @ m)**2)\nprint(f\"alpha = {alpha:.4f}, sigma derau = {1/np.sqrt(beta):.2f} ribu Rp, \"\n      f\"gamma = {gamma:.2f} dari {D} parameter\")\n\ndef prediksi(t_baru):                             # distribusi prediktif  #<2>\n    phi = fitur(np.atleast_1d(t_baru))\n    return phi @ m, np.sqrt(1/beta + np.sum(phi @ S * phi, axis=1))\n\nuji = ~latih\nr, s = prediksi(t[uji])\ncakup = np.mean(np.abs(y[uji] - r) < 1.96 * s)    # cakupan selang 95%  #<3>\nprint(f\"tahun 2025: {cakup:.1%} pengamatan jatuh di selang prediksi 95%\")\n\nprint(\"minggu           prediksi   +-2 sd   sebenarnya\")\nfor nama, tb in [(\"Lebaran 2025\", 169), (\"Juli 2025\", 182),\n                 (\"Juli 2027\", 286), (\"Juli 2032\", 546)]:            #<4>\n    r, s = prediksi(tb)\n    print(f\"{nama:14s} {r[0]:10.1f} {2*s[0]:8.1f} {pola_harga(tb):12.1f}\")\n",
"bab04/bias_variansi.py": "import numpy as np\n\nrng = np.random.default_rng(0)\nf = lambda x: 10 * np.sin(2 * np.pi * x)        # pola musiman sebenarnya\npusat = np.linspace(0, 1, 24)\ndef Phi(x):                                     # 24 basis Gaussian + konstanta\n    return np.column_stack([np.ones_like(x),\n                            np.exp(-(x[:, None] - pusat)**2 / (2 * 0.1**2))])\n\nx_uji = np.linspace(0, 1, 500)\nL, N = 100, 25                                  # 100 \"pasar\", 25 pengamatan  #<1>\nprint(\" ln(lambda)   bias^2   variansi   jumlah   galat uji\")\nfor ln_lam in (-6, -3, -0.3, 1, 3):\n    pred = np.empty((L, x_uji.size))\n    galat = []\n    for l in range(L):\n        x = rng.uniform(0, 1, N)\n        t = f(x) + rng.normal(0, 3, N)\n        A = np.exp(ln_lam) * np.eye(25) + Phi(x).T @ Phi(x)\n        w = np.linalg.solve(A, Phi(x).T @ t)        # ridge  #<2>\n        pred[l] = Phi(x_uji) @ w\n        galat.append(np.mean((pred[l] - f(x_uji) - rng.normal(0, 3, x_uji.size))**2))\n    rerata = pred.mean(axis=0)                      # model \"rata-rata\"  #<3>\n    bias2 = np.mean((rerata - f(x_uji))**2)\n    var = np.mean(pred.var(axis=0))\n    print(f\"{ln_lam:10.1f} {bias2:8.2f} {var:10.2f} {bias2 + var:8.2f} {np.mean(galat):11.2f}\")\n",
"bab04/data_harga.py": "\"\"\"Data sintetis harga cabai merah mingguan (ilustrasi, bukan data resmi).\"\"\"\nimport numpy as np\n\nMINGGU_LEBARAN = [18, 16+52, 15+104, 13+156]      # Idulfitri 2022-2025\n\ndef pola_harga(t):\n    \"\"\"Harga 'sebenarnya' (ribu Rp/kg) pada minggu ke-t sejak Januari 2022.\"\"\"\n    tren = 40 + 0.03 * t                           # naik pelan\n    musim = 9 * np.cos(2*np.pi*t/52) + 3 * np.sin(4*np.pi*t/52)  # puncak musim hujan\n    lebaran = sum(8 * np.exp(-0.5 * ((t - L) / 1.5)**2) for L in MINGGU_LEBARAN)\n    return tren + musim + lebaran\n\ndef buat_data(seed=0):\n    rng = np.random.default_rng(seed)\n    t = np.arange(208)                             # 4 tahun, mingguan\n    y = pola_harga(t) + rng.normal(0, 3.0, t.size) # derau pasar, sd Rp3.000\n    return t, y\n\ndef fitur(t, lebaran=True, tren=True, K=2):\n    \"\"\"Fungsi basis: konstanta, tren, Fourier tahunan, benjolan Lebaran.\"\"\"\n    kol = [np.ones_like(t, dtype=float)]\n    if tren:\n        kol.append(t / 52)\n    for k in range(1, K + 1):\n        kol += [np.cos(2*np.pi*k*t/52), np.sin(2*np.pi*k*t/52)]\n    if lebaran:\n        kol.append(sum(np.exp(-0.5*((t - L)/1.5)**2) for L in MINGGU_LEBARAN))\n    return np.column_stack(kol)\n",
"bab04/evidence_orde.py": "import numpy as np\n\nrng = np.random.default_rng(7)                    # data curah hujan Bab 1\nf = lambda x: 200 + 150 * np.cos(2 * np.pi * x)\nx = rng.uniform(0, 1, 12); t = f(x) + rng.normal(0, 40, 12)\nx_uji = rng.uniform(0, 1, 1000); t_uji = f(x_uji) + rng.normal(0, 40, 1000)\n\ndef evidence(M, iterasi=500):\n    Phi = np.vander(x, M + 1, increasing=True)\n    N, D = Phi.shape\n    lam0 = np.linalg.eigvalsh(Phi.T @ Phi)        # nilai eigen, dihitung sekali\n    alpha, beta = 1e-3, 1e-3\n    for _ in range(iterasi):                      # re-estimasi titik tetap  #<1>\n        A = alpha * np.eye(D) + beta * Phi.T @ Phi\n        m = beta * np.linalg.solve(A, Phi.T @ t)\n        lam = beta * lam0\n        gamma = np.sum(lam / (lam + alpha))       # banyak parameter efektif\n        alpha = gamma / (m @ m)\n        beta = (N - gamma) / np.sum((t - Phi @ m)**2)\n    A = alpha * np.eye(D) + beta * Phi.T @ Phi\n    m = beta * np.linalg.solve(A, Phi.T @ t)\n    E = beta/2 * np.sum((t - Phi @ m)**2) + alpha/2 * m @ m\n    ln_ev = (D/2*np.log(alpha) + N/2*np.log(beta) - E                 #<2>\n             - np.linalg.slogdet(A)[1]/2 - N/2*np.log(2*np.pi))\n    rmse = np.sqrt(np.mean((np.vander(x_uji, D, increasing=True) @ m - t_uji)**2))\n    return ln_ev, gamma, 1/np.sqrt(beta), rmse\n\nprint(\" M   ln evidence   gamma   sigma (mm)   RMSE uji (mm)\")\nfor M in range(10):\n    ln_ev, g, s, r = evidence(M)\n    print(f\"{M:2d} {ln_ev:13.2f} {g:7.2f} {s:12.1f} {r:15.1f}\")\n",
"bab04/lasso_sparsitas.py": "import numpy as np\nfrom sklearn.linear_model import Lasso, Ridge\nfrom sklearn.preprocessing import StandardScaler\nfrom data_harga import buat_data, fitur\n\nt, y = buat_data()\nlatih, uji = t < 156, t >= 156\nrng = np.random.default_rng(1)\nacak = rng.normal(size=(t.size, 20)).cumsum(axis=0)   # 20 deret tak berkaitan  #<1>\nX = np.column_stack([fitur(t, K=6)[:, 1:], acak])      # 14 fitur asli + 20 acak\nnama_fitur = ([\"tren\"] + [f\"{f}{k}\" for k in range(1, 7) for f in (\"cos\", \"sin\")]\n              + [\"Lebaran\"] + [f\"acak{j}\" for j in range(20)])\nskala = StandardScaler().fit(X[latih])                  #<2>\nX = skala.transform(X)\n\nprint(f\"{'model':16s} {'bukan nol':>9s} {'dari acak':>9s} {'RMSE uji':>9s}\")\nfor nama, m in [(\"kuadrat terkecil\", Ridge(alpha=1e-9)), (\"ridge a=10\", Ridge(alpha=10)),\n                (\"lasso a=0.05\", Lasso(alpha=0.05, max_iter=50_000)),\n                (\"lasso a=0.4\", Lasso(alpha=0.4, max_iter=50_000)),         #<3>\n                (\"lasso a=1.5\", Lasso(alpha=1.5, max_iter=50_000))]:\n    m.fit(X[latih], y[latih])\n    nz = np.abs(m.coef_) > 1e-6\n    rmse = np.sqrt(np.mean((m.predict(X[uji]) - y[uji])**2))\n    print(f\"{nama:16s} {nz.sum():9d} {nz[14:].sum():9d} {rmse:9.2f}\")\n    if nama == \"lasso a=0.4\":\n        terpilih = [n for n, z in zip(nama_fitur, nz) if z]\n\nprint(\"lasso a=0.4 memilih:\", \", \".join(terpilih))\n"
}
akar = pathlib.Path('pola-ml/kode').resolve()
for nama, isi in BERKAS.items():
    p = akar / nama; p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(isi, encoding='utf-8')
os.chdir(akar / 'bab04')
print('folder kerja:', os.getcwd())

## Listing 4.1: Membandingkan fungsi basis untuk harga cabai mingguan

Berkas: `kode/bab04/basis_harga.py`

In [ ]:
import numpy as np
from data_harga import buat_data, fitur

t, y = buat_data()
latih, uji = t < 156, t >= 156                     # 3 tahun latih, 2025 uji  # (1)

def rmse(Phi):
    w, *_ = np.linalg.lstsq(Phi[latih], y[latih], rcond=None)  # (2)
    galat = Phi @ w - y
    return np.sqrt(np.mean(galat[latih]**2)), np.sqrt(np.mean(galat[uji]**2))

s = (t - 104) / 104                                # skala [-1, 1] agar stabil
model = {
    "polinom orde 9":         np.vander(s, 10, increasing=True),   # (3)
    "Fourier saja":           fitur(t, lebaran=False, tren=False),
    "Fourier + tren":         fitur(t, lebaran=False),
    "Fourier + tren + Lebaran": fitur(t),                          # (4)
}
print(f"{'model':26s} {'#param':>6s} {'RMSE latih':>11s} {'RMSE uji':>9s}")
for nama, Phi in model.items():
    r_l, r_u = rmse(Phi)
    print(f"{nama:26s} {Phi.shape[1]:6d} {r_l:11.2f} {r_u:9.2f}")

## Listing 4.2: Lasso sebagai pemilih fitur, diuji dengan fitur pengecoh

Berkas: `kode/bab04/lasso_sparsitas.py`

In [ ]:
import numpy as np
from sklearn.linear_model import Lasso, Ridge
from sklearn.preprocessing import StandardScaler
from data_harga import buat_data, fitur

t, y = buat_data()
latih, uji = t < 156, t >= 156
rng = np.random.default_rng(1)
acak = rng.normal(size=(t.size, 20)).cumsum(axis=0)   # 20 deret tak berkaitan  # (1)
X = np.column_stack([fitur(t, K=6)[:, 1:], acak])      # 14 fitur asli + 20 acak
nama_fitur = (["tren"] + [f"{f}{k}" for k in range(1, 7) for f in ("cos", "sin")]
              + ["Lebaran"] + [f"acak{j}" for j in range(20)])
skala = StandardScaler().fit(X[latih])                  # (2)
X = skala.transform(X)

print(f"{'model':16s} {'bukan nol':>9s} {'dari acak':>9s} {'RMSE uji':>9s}")
for nama, m in [("kuadrat terkecil", Ridge(alpha=1e-9)), ("ridge a=10", Ridge(alpha=10)),
                ("lasso a=0.05", Lasso(alpha=0.05, max_iter=50_000)),
                ("lasso a=0.4", Lasso(alpha=0.4, max_iter=50_000)),         # (3)
                ("lasso a=1.5", Lasso(alpha=1.5, max_iter=50_000))]:
    m.fit(X[latih], y[latih])
    nz = np.abs(m.coef_) > 1e-6
    rmse = np.sqrt(np.mean((m.predict(X[uji]) - y[uji])**2))
    print(f"{nama:16s} {nz.sum():9d} {nz[14:].sum():9d} {rmse:9.2f}")
    if nama == "lasso a=0.4":
        terpilih = [n for n, z in zip(nama_fitur, nz) if z]

print("lasso a=0.4 memilih:", ", ".join(terpilih))

## Listing 4.3: Mengukur bias dan variansi dengan 100 set data

Berkas: `kode/bab04/bias_variansi.py`

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
f = lambda x: 10 * np.sin(2 * np.pi * x)        # pola musiman sebenarnya
pusat = np.linspace(0, 1, 24)
def Phi(x):                                     # 24 basis Gaussian + konstanta
    return np.column_stack([np.ones_like(x),
                            np.exp(-(x[:, None] - pusat)**2 / (2 * 0.1**2))])

x_uji = np.linspace(0, 1, 500)
L, N = 100, 25                                  # 100 "pasar", 25 pengamatan  # (1)
print(" ln(lambda)   bias^2   variansi   jumlah   galat uji")
for ln_lam in (-6, -3, -0.3, 1, 3):
    pred = np.empty((L, x_uji.size))
    galat = []
    for l in range(L):
        x = rng.uniform(0, 1, N)
        t = f(x) + rng.normal(0, 3, N)
        A = np.exp(ln_lam) * np.eye(25) + Phi(x).T @ Phi(x)
        w = np.linalg.solve(A, Phi(x).T @ t)        # ridge  # (2)
        pred[l] = Phi(x_uji) @ w
        galat.append(np.mean((pred[l] - f(x_uji) - rng.normal(0, 3, x_uji.size))**2))
    rerata = pred.mean(axis=0)                      # model "rata-rata"  # (3)
    bias2 = np.mean((rerata - f(x_uji))**2)
    var = np.mean(pred.var(axis=0))
    print(f"{ln_lam:10.1f} {bias2:8.2f} {var:10.2f} {bias2 + var:8.2f} {np.mean(galat):11.2f}")

## Listing 4.4: Regresi linear Bayesian untuk harga cabai, dengan $$ dan $$ dari data

Berkas: `kode/bab04/bayes_harga.py`

In [ ]:
import numpy as np
from data_harga import buat_data, fitur, pola_harga

t, y = buat_data()
latih = t < 156
Phi, tl = fitur(t[latih]), y[latih]
N, D = Phi.shape

alpha, beta = 1.0, 1.0
for _ in range(100):                              # alpha, beta dari evidence  # (1)
    S = np.linalg.inv(alpha * np.eye(D) + beta * Phi.T @ Phi)
    m = beta * S @ Phi.T @ tl
    gamma = D - alpha * np.trace(S)
    alpha, beta = gamma / (m @ m), (N - gamma) / np.sum((tl - Phi @ m)**2)
print(f"alpha = {alpha:.4f}, sigma derau = {1/np.sqrt(beta):.2f} ribu Rp, "
      f"gamma = {gamma:.2f} dari {D} parameter")

def prediksi(t_baru):                             # distribusi prediktif  # (2)
    phi = fitur(np.atleast_1d(t_baru))
    return phi @ m, np.sqrt(1/beta + np.sum(phi @ S * phi, axis=1))

uji = ~latih
r, s = prediksi(t[uji])
cakup = np.mean(np.abs(y[uji] - r) < 1.96 * s)    # cakupan selang 95%  # (3)
print(f"tahun 2025: {cakup:.1%} pengamatan jatuh di selang prediksi 95%")

print("minggu           prediksi   +-2 sd   sebenarnya")
for nama, tb in [("Lebaran 2025", 169), ("Juli 2025", 182),
                 ("Juli 2027", 286), ("Juli 2032", 546)]:            # (4)
    r, s = prediksi(tb)
    print(f"{nama:14s} {r[0]:10.1f} {2*s[0]:8.1f} {pola_harga(tb):12.1f}")

## Listing 4.5: Evidence dan parameter efektif untuk orde polinom 0–9

Berkas: `kode/bab04/evidence_orde.py`

In [ ]:
import numpy as np

rng = np.random.default_rng(7)                    # data curah hujan Bab 1
f = lambda x: 200 + 150 * np.cos(2 * np.pi * x)
x = rng.uniform(0, 1, 12); t = f(x) + rng.normal(0, 40, 12)
x_uji = rng.uniform(0, 1, 1000); t_uji = f(x_uji) + rng.normal(0, 40, 1000)

def evidence(M, iterasi=500):
    Phi = np.vander(x, M + 1, increasing=True)
    N, D = Phi.shape
    lam0 = np.linalg.eigvalsh(Phi.T @ Phi)        # nilai eigen, dihitung sekali
    alpha, beta = 1e-3, 1e-3
    for _ in range(iterasi):                      # re-estimasi titik tetap  # (1)
        A = alpha * np.eye(D) + beta * Phi.T @ Phi
        m = beta * np.linalg.solve(A, Phi.T @ t)
        lam = beta * lam0
        gamma = np.sum(lam / (lam + alpha))       # banyak parameter efektif
        alpha = gamma / (m @ m)
        beta = (N - gamma) / np.sum((t - Phi @ m)**2)
    A = alpha * np.eye(D) + beta * Phi.T @ Phi
    m = beta * np.linalg.solve(A, Phi.T @ t)
    E = beta/2 * np.sum((t - Phi @ m)**2) + alpha/2 * m @ m
    ln_ev = (D/2*np.log(alpha) + N/2*np.log(beta) - E                 # (2)
             - np.linalg.slogdet(A)[1]/2 - N/2*np.log(2*np.pi))
    rmse = np.sqrt(np.mean((np.vander(x_uji, D, increasing=True) @ m - t_uji)**2))
    return ln_ev, gamma, 1/np.sqrt(beta), rmse

print(" M   ln evidence   gamma   sigma (mm)   RMSE uji (mm)")
for M in range(10):
    ln_ev, g, s, r = evidence(M)
    print(f"{M:2d} {ln_ev:13.2f} {g:7.2f} {s:12.1f} {r:15.1f}")